In [10]:
!apt-get update -qq
!apt-get install -y flex bison gcc

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
bison is already the newest version (2:3.8.2+dfsg-1build1).
flex is already the newest version (2.6.4-8build2).
gcc is already the newest version (4:11.2.0-1ubuntu1).
0 upgraded, 0 newly installed, 0 to remove and 16 not upgraded.


In [11]:
%%writefile control.l
%{
#include "control.tab.h"
%}

%%

"if"        { return IF; }
"else"      { return ELSE; }
"for"       { return FOR; }
"while"     { return WHILE; }
"switch"    { return SWITCH; }
"case"      { return CASE; }
"default"   { return DEFAULT; }

[a-zA-Z_][a-zA-Z0-9_]*   { return ID; }
[0-9]+                   { return NUM; }

"{"         { return LBRACE; }
"}"         { return RBRACE; }
"("         { return LPAREN; }
")"         { return RPAREN; }
":"         { return COLON; }
";"         { return SEMICOLON; }

"=="        { return EQ; }
"<="        { return LE; }
">="        { return GE; }
"<"         { return LT; }
">"         { return GT; }
"="         { return ASSIGN; }

[ \t\r\n]+  { /* ignore whitespace */ }

.           { return yytext[0]; }

%%

int yywrap()
{
    return 1;
}

Overwriting control.l


In [12]:
%%writefile control.y
%{
#include <stdio.h>
#include <stdlib.h>

int yylex(void);
int yyerror(const char *s);
%}

%token IF ELSE FOR WHILE SWITCH CASE DEFAULT
%token ID NUM
%token LBRACE RBRACE LPAREN RPAREN COLON SEMICOLON
%token EQ LE GE LT GT ASSIGN

%%

program
    : stmt_list
    ;

stmt_list
    : stmt_list stmt
    | stmt
    ;

stmt
    : if_stmt
    | while_stmt
    | for_stmt
    | switch_stmt
    | simple_stmt
    ;

simple_stmt
    : ID ASSIGN NUM SEMICOLON
    | ID ASSIGN ID SEMICOLON
    ;

if_stmt
    : IF LPAREN cond RPAREN stmt
    | IF LPAREN cond RPAREN stmt ELSE stmt
    ;

while_stmt
    : WHILE LPAREN cond RPAREN stmt
    ;

for_stmt
    : FOR LPAREN ID ASSIGN NUM SEMICOLON cond SEMICOLON ID ASSIGN ID RPAREN stmt
    ;

switch_stmt
    : SWITCH LPAREN ID RPAREN LBRACE case_list RBRACE
    ;

case_list
    : case_list CASE NUM COLON stmt
    | case_list DEFAULT COLON stmt
    | CASE NUM COLON stmt
    | DEFAULT COLON stmt
    ;

cond
    : ID relop NUM
    ;

relop
    : EQ
    | LE
    | GE
    | LT
    | GT
    ;

%%

int main()
{
    printf("Enter C control structure:\n");

    if (yyparse() == 0)
        printf("Valid control structure syntax.\n");

    return 0;
}

int yyerror(const char *s)
{
    printf("Invalid control structure syntax.\n");
    return 0;
}

Overwriting control.y


In [13]:
!bison -d control.y

control.y: warning: 1 shift/reduce conflict [-Wconflicts-sr]
control.y: note: rerun with option '-Wcounterexamples' to generate conflict counterexamples


In [14]:
!flex control.l

In [15]:
!gcc lex.yy.c control.tab.c -o control -lfl

In [16]:
!echo "if(a<10) b=5;" | ./control

Enter C control structure:
Valid control structure syntax.


In [17]:
!echo "if(a<10) b=5; else b=10;" | ./control

Enter C control structure:
Valid control structure syntax.


In [18]:
!echo "while(a<10) b=5;" | ./control

Enter C control structure:
Valid control structure syntax.
